<a href="https://colab.research.google.com/github/icezac999/PI_Mineria_Datos_1/blob/main/notebooks/01_inspeccion_inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hito 1 — Inspección Inicial y Calidad de Datos

**Proyecto Integrador — Minería de Datos 1**

**Dataset:** `streaming_users_dirty.json` (usuarios de una plataforma de streaming)

## Objetivo de este notebook
Antes de limpiar o analizar los datos, necesitamos entender **qué tenemos**: cuántos registros hay, qué columnas existen, y sobre todo, **qué problemas de calidad** presenta el dataset. Esta inspección es la base que va a justificar las decisiones de limpieza que tomemos en el Hito 2.

No vamos a corregir nada todavía. Solo vamos a **observar y documentar**.


## 1. Carga de datos

In [20]:
from google.colab import files
uploaded = files.upload()

In [21]:
import pandas as pd
import numpy as np

df = pd.read_json("streaming_users_dirty.json")
df.head()


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## 2. Estructura general del dataset

Vemos cuántas filas y columnas tiene, qué tipo de dato tiene cada columna, y un resumen estadístico rápido.

In [22]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.dtypes


Filas: 8160
Columnas: 8


,0
user_id,int64
age,int64
subscription_plan,object
monthly_watch_time_mins,float64
country,object
favorite_genre,object
last_login_date,object
customer_support_tickets,int64


In [23]:
df.describe(include="all")


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
count,8160.000000,8160.000000,8160,7967.000000,8160,7920,7840,8160.000000
unique,NaN,NaN,15,NaN,26,28,3062,NaN
top,NaN,NaN,Básico,NaN,Brasil,Comedia,2026-15-03,NaN
freq,NaN,NaN,3450,NaN,1132,1112,20,NaN
mean,13995.433824,34.096814,NaN,1107.346153,NaN,NaN,NaN,1.800980
std,2310.810660,14.511304,NaN,5310.442604,NaN,NaN,NaN,11.334969
min,10000.000000,-5.000000,NaN,-120.000000,NaN,NaN,NaN,-1.000000
25%,11987.750000,25.000000,NaN,489.200000,NaN,NaN,NaN,0.000000
50%,13998.500000,33.000000,NaN,757.400000,NaN,NaN,NaN,1.000000
75%,15997.250000,42.000000,NaN,1045.700000,NaN,NaN,NaN,1.000000


**Columnas del dataset:**

| Columna | Descripción |
|---|---|
| `user_id` | Identificador único del usuario |
| `age` | Edad del usuario |
| `subscription_plan` | Plan de suscripción (Básico, Estándar, Premium) |
| `monthly_watch_time_mins` | Minutos de consumo mensual |
| `country` | País del usuario |
| `favorite_genre` | Género favorito |
| `last_login_date` | Fecha del último ingreso |
| `customer_support_tickets` | Cantidad de tickets de soporte generados |


## 3. Valores nulos

In [24]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "% del total": porcentaje})


,nulos,% del total
user_id,0,0.00
age,0,0.00
subscription_plan,0,0.00
monthly_watch_time_mins,193,2.37
country,0,0.00
favorite_genre,240,2.94
last_login_date,320,3.92
customer_support_tickets,0,0.00


**Hallazgo:** `favorite_genre` y `last_login_date` presentan valores nulos. Las demás columnas están completas. Esto ya nos anticipa que en el Hito 2 vamos a tener que decidir cómo tratar estos faltantes (imputar, dejar como categoría aparte, etc.).

## 4. Registros duplicados

In [25]:
# Duplicados exactos (toda la fila igual)
duplicados_totales = df.duplicated().sum()
print("Filas totalmente duplicadas:", duplicados_totales)

# Duplicados por user_id (el identificador no debería repetirse)
duplicados_id = df["user_id"].duplicated().sum()
print("user_id repetidos:", duplicados_id)


Filas totalmente duplicadas: 126
user_id repetidos: 160


**Hallazgo:** Hay filas completamente duplicadas y además `user_id` se repite más veces de las que se explican solo por esas filas duplicadas. Esto sugiere que también existen usuarios cargados más de una vez con datos distintos (posible error de carga). Es un punto a resolver en el Hito 2.

## 5. Inconsistencias en variables categóricas

In [26]:
df["subscription_plan"].value_counts()


,count
subscription_plan,
Básico,3450
Estándar,2711
Premium,1519
basico,60
BASICO,52
Basic,52
básico,50
Std,48
Estándar,46


In [27]:
df["country"].value_counts()


,count
country,
Brasil,1132
Chile,1132
México,1129
Uruguay,1124
Perú,1120
Colombia,1116
Argentina,1087
colombia,27
méxico,25


In [28]:
df["favorite_genre"].value_counts(dropna=False)


,count
favorite_genre,
Comedia,1112
Drama,1105
Documental,1095
Romance,1090
Thriller,1090
Acción,1082
Crime,1067
None,240
Action,22


**Hallazgo:** Las tres columnas categóricas tienen el mismo problema: **la misma categoría está escrita de distintas formas**.

- `subscription_plan`: por ejemplo "Básico", "basico", "BASICO", "Basic" y "Std" parecen representar el mismo plan.
- `country`: por ejemplo "Colombia", "colombia" y "COL" parecen ser el mismo país.
- `favorite_genre`: por ejemplo "Comedia", "COMEDIA", "comedy" y "Comedia " (con espacio) parecen ser el mismo género.

Esto es típico de un dataset con errores de tipeo, uso de mayúsculas/minúsculas mezcladas, abreviaturas y espacios extra. Se deberá estandarizar en el Hito 2.

## 6. Valores fuera de rango (posibles outliers o errores)

In [29]:
print("Edad — mínimo:", df["age"].min(), " máximo:", df["age"].max())
print("Minutos de consumo — mínimo:", df["monthly_watch_time_mins"].min(), " máximo:", df["monthly_watch_time_mins"].max())
print("Tickets de soporte — mínimo:", df["customer_support_tickets"].min(), " máximo:", df["customer_support_tickets"].max())


Edad — mínimo: -5  máximo: 150
Minutos de consumo — mínimo: -120.0  máximo: 99999.0
Tickets de soporte — mínimo: -1  máximo: 150


In [30]:
# Cantidad de registros con valores claramente imposibles
print("Edades negativas o mayores a 100:", ((df["age"] < 0) | (df["age"] > 100)).sum())
print("Minutos de consumo negativos:", (df["monthly_watch_time_mins"] < 0).sum())
print("Tickets de soporte negativos:", (df["customer_support_tickets"] < 0).sum())


Edades negativas o mayores a 100: 74
Minutos de consumo negativos: 49
Tickets de soporte negativos: 29


**Hallazgo:** Encontramos valores que no tienen sentido en el mundo real:
- Edades negativas y edades de 150 años.
- Minutos de consumo negativos y un valor extremo de 99999 minutos (~69 días de consumo en un mes, imposible).
- Tickets de soporte negativos.

Estos son errores de carga o de sensor, no representan usuarios reales. Se deberán tratar en el Hito 2 (ya sea eliminando esos registros o corrigiéndolos si hay evidencia suficiente).

## 7. Formato de fechas

In [31]:
import re

fechas = df["last_login_date"].dropna()

formato_estandar = fechas.str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas en formato AAAA-MM-DD:", formato_estandar.sum())
print("Fechas en otro formato:", (~formato_estandar).sum())

fechas[~formato_estandar].head(10)


Fechas en formato AAAA-MM-DD: 7428
Fechas en otro formato: 412


,last_login_date
53,11-19-2018
64,01-06-2025
81,11-14-2025
148,03-16-2024
155,2025/03/10
160,2019/10/10
170,2025/03/02
178,12-13-2023
219,11-05-2019
243,2025/07/03


**Hallazgo:** La mayoría de las fechas siguen el formato `AAAA-MM-DD`, pero hay un grupo importante en otros formatos (`MM-DD-AAAA`, `AAAA/MM/DD`). Antes de poder trabajar con estas fechas como tipo fecha real, hay que unificar el formato.

## 8. Resumen de hallazgos — Calidad inicial del dataset

| Problema detectado | Columna(s) afectada(s) | Impacto |
|---|---|---|
| Valores nulos | `favorite_genre`, `last_login_date` | Requiere decisión de imputación o categoría "sin dato" |
| Filas y `user_id` duplicados | Todo el dataset | Puede inflar el análisis si no se eliminan |
| Categorías escritas de forma inconsistente | `subscription_plan`, `country`, `favorite_genre` | Impide agrupar correctamente sin estandarizar |
| Valores fuera de rango / imposibles | `age`, `monthly_watch_time_mins`, `customer_support_tickets` | Distorsionan estadísticas y análisis si no se tratan |
| Formatos de fecha mixtos | `last_login_date` | Impide convertir la columna a tipo fecha directamente |

Estas observaciones, basadas en evidencia concreta del dataset, son las que van a justificar cada decisión de limpieza que tomemos en el **Hito 2 (Preparación de datos)**. No se corrige nada en este notebook: el objetivo acá era únicamente **documentar el estado real del dataset**.
